# 02. AI 플레이어 1명 단위 테스트

**목적**: 전체 게임을 돌리기 전에 AI 플레이어 한 명씩 제대로 동작하는지 확인한다.

확인 항목
1. 역할별로 `LLMPlayer`를 하나씩 만든다 (마피아 / 경찰 / 의사 / 시민)
2. **정보 분리**: AI에게 전달되는 `PlayerView`에는 자기가 알아도 되는 정보만 들어있다
3. 발언 1회, 투표 1회, 밤 행동(처치/보호/조사)
4. **정체 노출 테스트**: 마피아가 "나 마피아야"라고 말하면 코드가 막는다
5. **fallback**: LLM이 엉뚱한 출력을 해도 게임이 멈추지 않는다

설계 원칙: *LLM은 연기를 하고, 진실(규칙/정보 접근권한)은 코드가 지킨다.*

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
.
LLM: qwen3:8b / 임베딩: bge-m3


In [2]:
# ===== 1. 모듈 불러오기 =====
import rag
import mafia_core as core
import mafia_agents as agents

vs = rag.load_db()
print("===== 벡터 DB 로드 완료 =====")
print("Chunk 수:", vs._collection.count())

===== 벡터 DB 로드 완료 =====
Chunk 수: 23


## 1. 게임 하나 만들고 역할별 AI 만들기

`new_game()`이 역할을 무작위로 배정한다 (seed를 주면 같은 결과). 이 상태(`GameState`)는 **진실의 원천**이며 AI에게 직접 전달하지 않는다.

In [3]:
# ===== 2. 게임 생성 =====
state = core.new_game(list(config.CHARACTERS), None, seed=7)

print("===== 역할 배정 (진실, AI에게는 보이지 않음) =====")
for p in state.players:
    print("  %s : %s" % (p.name, config.ROLE_KO[p.role]))

# 역할별 대표 플레이어 한 명씩 고르기
rep = {}
for p in state.players:
    rep.setdefault(p.role, p.name)
print("\n대표 플레이어:", {config.ROLE_KO[r]: n for r, n in rep.items()})

===== 역할 배정 (진실, AI에게는 보이지 않음) =====
  강태호 : 시민
  윤서진 : 시민
  박준수 : 경찰
  최민지 : 시민
  정도현 : 마피아
  한소율 : 의사
  오지환 : 마피아
  임채원 : 시민

대표 플레이어: {'시민': '강태호', '경찰': '박준수', '마피아': '정도현', '의사': '한소율'}


In [4]:
# ===== 3. 역할별 LLMPlayer 생성 =====
llm = agents.make_llm()

players = {
    role: agents.LLMPlayer(name, role, vs, llm=llm, state=state)
    for role, name in rep.items()
}

print("===== LLMPlayer 생성 완료 =====")
for role, ai in players.items():
    print("%-6s %-5s 공통 도구: %s" % (config.ROLE_KO[role], ai.name, list(ai.tools.keys())))
print("(역할별 행동 도구 - 마피아 choose_kill_target / 의사 protect / 경찰 investigate - 는 행동할 때마다 후보 목록과 함께 만들어진다)")

===== LLMPlayer 생성 완료 =====
시민     강태호   공통 도구: ['search_strategy', 'read_my_notes']
경찰     박준수   공통 도구: ['search_strategy', 'read_my_notes']
마피아    정도현   공통 도구: ['search_strategy', 'read_my_notes']
의사     한소율   공통 도구: ['search_strategy', 'read_my_notes']
(역할별 행동 도구 - 마피아 choose_kill_target / 의사 protect / 경찰 investigate - 는 행동할 때마다 후보 목록과 함께 만들어진다)


## 2. 정보 분리 확인: PlayerView

AI에게는 `build_view()`가 만든 `PlayerView`만 전달된다. 같은 시점에 **마피아의 시야**와 **시민의 시야**를 비교해보자.
마피아의 `private_info`에는 동료 마피아가 있지만, 시민에게는 아무것도 없어야 한다.

In [5]:
# ===== 4. PlayerView 비교 =====
mafia_name = rep["mafia"]
citizen_name = rep["citizen"]

view_m = core.build_view(state, mafia_name, "speak")
view_c = core.build_view(state, citizen_name, "speak")

for v in [view_m, view_c]:
    print("----- %s의 PlayerView -----" % v.name)
    print("자기 역할:", v.role_ko)
    print("비공개 정보(private_info):", v.private_info)
    print("투표/발언 대상 후보:", v.candidates)
print()
print("마피아 view 에 동료 정보가 있는가?", len(view_m.private_info) > 0)
print("시민 view 에 비공개 정보가 있는가?", len(view_c.private_info) > 0)
print("PlayerView 필드 목록:", list(view_m.__dataclass_fields__.keys()))
print("-> 다른 사람의 역할(role) 필드는 PlayerView 어디에도 없다.")

----- 정도현의 PlayerView -----
자기 역할: 마피아
비공개 정보(private_info): ['동료 마피아: 오지환']
투표/발언 대상 후보: ['강태호', '윤서진', '박준수', '최민지', '한소율', '오지환', '임채원']
----- 강태호의 PlayerView -----
자기 역할: 시민
비공개 정보(private_info): []
투표/발언 대상 후보: ['윤서진', '박준수', '최민지', '정도현', '한소율', '오지환', '임채원']

마피아 view 에 동료 정보가 있는가? True
시민 view 에 비공개 정보가 있는가? False
PlayerView 필드 목록: ['name', 'role', 'role_ko', 'day', 'phase', 'alive', 'dead', 'public_log_text', 'private_info', 'my_notes', 'candidates', 'partner_proposal']
-> 다른 사람의 역할(role) 필드는 PlayerView 어디에도 없다.


## 3. 밤 행동 (역할별 도구)

- 마피아: `choose_kill_target` 으로 처치 대상 선택
- 의사: `protect` 로 보호 대상 선택
- 경찰: `investigate` 로 조사 대상 선택
- 시민: 밤 행동 없음 (`None`)

AI는 도구로 선택을 **기록**할 뿐이고, 선택이 유효한지는 코드가 다시 검사한다. `last_source`는 어떤 경로로 답이 나왔는지 알려준다 (agent / structured / none).

In [6]:
# ===== 5. 밤 행동 =====
actions = {"mafia": "kill", "doctor": "protect", "police": "investigate"}

for role, action in actions.items():
    ai = players[role]
    view = core.build_view(state, ai.name, action)
    decision = ai.night_action(view)
    print("[%s %s] 후보: %s" % (config.ROLE_KO[role], ai.name, view.candidates))
    print("   선택:", decision.target if decision else None, "| 이유:", getattr(decision, "reason", ""), "| 경로:", ai.last_source)

print("\n[시민]", players["citizen"].night_action(core.build_view(state, rep["citizen"], "speak")))

[마피아 정도현] 후보: ['강태호', '윤서진', '박준수', '최민지', '한소율', '임채원']
   선택: 박준수 | 이유: 박준수는 논리적으로 추리해 주목받는 사람으로 보이며, 시민팀에 위협이 될 수 있는 인물입니다. | 경로: agent
[의사 한소율] 후보: ['강태호', '윤서진', '박준수', '최민지', '정도현', '한소율', '오지환', '임채원']
   선택: 한소율 | 이유: 자신을 보호하여 마피아의 공격을 막고, 이후의 추리와 판단에 유리하게 만든다. | 경로: agent
[경찰 박준수] 후보: ['강태호', '윤서진', '최민지', '정도현', '한소율', '오지환', '임채원']
   선택: 임채원 | 이유: 임채원이 가장 수상하고 영향력이 있는 사람으로 판단됩니다. | 경로: agent

[시민] None


## 4. 발언 1회와 투표 1회

발언은 한 번의 LLM 호출로 `{속마음(thought), 의심도(suspicion), 발언(text)}`을 구조화해서 받는다.
**속마음은 게임 중에는 공개되지 않고** `state.notes`에만 저장된다 (게임이 끝나면 `reveal_report`로 공개).

In [7]:
# ===== 6. 발언 1회 =====
import time

ai = players["citizen"]
view = core.build_view(state, ai.name, "speak")

t0 = time.time()
speech = ai.speak(view)
print("[%s] 발언: %s" % (ai.name, speech.text))
print("   속마음(비공개):", speech.thought)
print("   의심도(비공개):", speech.suspicion)
print("   소요 시간: %.1f초" % (time.time() - t0))

[강태호] 발언: 야, 다들 말 아끼지 마. 오늘은 일단 얘기부터 들어보자. 나야 진짜 의심이 들면 바로 말해. 지금까지 아무도 말 안 했으니까, 나부터 물어볼게. 윤서진, 너 처음부터 왜 이렇게 말을 잘하니? 이상한 거 없어?
   속마음(비공개): 야, 다들 말 아끼지 마. 오늘은 일단 얘기부터 들어보자. 나야 진짜 의심이 들면 바로 말해. 지금까지 아무도 말 안 했으니까, 나부터 물어볼게. 윤서진, 너 처음부터 왜 이렇게 말을 잘하니? 이상한 거 없어?
   의심도(비공개): {'윤서진': 30, '박준수': 10, '최민지': 5, '정도현': 5, '한소율': 5, '오지환': 5, '임채원': 5}
   소요 시간: 5.4초


In [8]:
# ===== 7. 투표 1회 =====
ai = players["police"]
view = core.build_view(state, ai.name, "vote")

vote = ai.vote(view)
print("[%s] 투표 후보: %s" % (ai.name, view.candidates))
print("   투표 -> %s (이유: %s)" % (vote.target, vote.reason))
print("   후보 안에 있는가?", core.normalize_choice(vote.target, view.candidates) is not None)

[박준수] 투표 후보: ['강태호', '윤서진', '최민지', '정도현', '한소율', '오지환', '임채원']
   투표 -> 강태호 (이유: 강태호가 가장 수상하고 영향력이 있는 인물로 판단되며, 마피아일 가능성이 높습니다.)
   후보 안에 있는가? True


## 5. 정체 노출 테스트 (leak test)

마피아가 공개 토론에서 "나는 마피아야" 같은 말을 하면 게임이 망가진다. 두 겹으로 방어한다.
1. 프롬프트에서 금지 (LLM이 지켜주길 기대)
2. **코드 검사** `contains_role_leak()` - LLM이 어겨도 코드가 걸러낸다 (이 노트북에서 확인)

In [9]:
# ===== 8-1. 규칙 검사 단위 테스트 =====
samples = [
    ("mafia", "나는 마피아야, 믿어줘"),
    ("mafia", "저는 마피아가 아닙니다. 시민이에요."),
    ("mafia", "우리 마피아끼리 힘내자"),
    ("mafia", "저는 윤서진님이 의심스럽습니다."),
    ("citizen", "저는 시민입니다."),
]
for role, text in samples:
    print("%-4s | 노출=%-5s | %s" % (role, core.contains_role_leak(text, role), text))

mafia | 노출=True  | 나는 마피아야, 믿어줘
mafia | 노출=False | 저는 마피아가 아닙니다. 시민이에요.
mafia | 노출=True  | 우리 마피아끼리 힘내자
mafia | 노출=False | 저는 윤서진님이 의심스럽습니다.
citizen | 노출=False | 저는 시민입니다.


In [10]:
# ===== 8-2. 노출 발언을 하는 가짜 마피아로 safe_speak 확인 =====
class LeakyMafia:
    """항상 정체를 말해버리는 나쁜 AI (테스트용)"""

    def speak(self, view):
        return core.Speech(text="나는 마피아야! 우리 마피아끼리 이기자!", thought="일부러 노출")


test_state = core.new_game(list(config.CHARACTERS), None, seed=1)
test_mafia = [p.name for p in test_state.players if p.role == "mafia"][0]
test_view = core.build_view(test_state, test_mafia, "speak")

speech, used_fallback = core.safe_speak(LeakyMafia().speak, test_view, test_state)
print("최종 발언:", speech.text)
print("fallback 사용:", used_fallback)
print("leak_blocked 통계:", test_state.stats["leak_blocked"])

최종 발언: 섣불리 의심하기보다 투표 흐름을 보고 싶습니다.
fallback 사용: True
leak_blocked 통계: 2


In [11]:
# ===== 8-3. 실제 LLM 마피아가 정체를 말하는지 3회 확인 =====
leaks = 0
mafia_ai = players["mafia"]
for i in range(3):
    sp = mafia_ai.speak(core.build_view(state, mafia_ai.name, "speak"))
    leaked = core.contains_role_leak(sp.text, "mafia")
    leaks += int(leaked)
    print("%d회: 노출=%s | %s" % (i + 1, leaked, sp.text))
print("정체 노출 횟수: %d / 3" % leaks)

1회: 노출=False | 어디서 그런 말들 들었어?
2회: 노출=False | 임채원, 왜 처음부터 조용했어? 말이 적은 게 왜 이상해?
3회: 노출=False | 강태호, 말이 너무 많아. 수상하다.
정체 노출 횟수: 0 / 3


## 6. 안전장치(fallback) 강제 시연

LLM이 없는 이름을 말하거나, 예외를 내거나, 엉뚱한 형식을 돌려줘도 `safe_decide()`가
**정규화 -> 재시도 -> 무작위 대체** 순서로 처리해서 게임은 절대 멈추지 않는다.

In [12]:
# ===== 9. fallback 시연 =====
class BrokenAI:
    """항상 이상한 출력을 내는 AI (테스트용)"""

    def __init__(self, mode):
        self.mode = mode

    def vote(self, view):
        if self.mode == "garbage":
            return core.VoteDecision("몰라요 아무나")      # 후보에 없는 이름
        if self.mode == "error":
            raise RuntimeError("LLM 호출 실패")           # 예외
        return core.VoteDecision("3번 " + view.candidates[2] + "님")   # 퍼지 매칭으로 구제 가능


demo_state = core.new_game(list(config.CHARACTERS), None, seed=3)
demo_view = core.build_view(demo_state, demo_state.players[0].name, "vote")

for mode in ["fuzzy", "garbage", "error"]:
    before = dict(demo_state.stats)
    target, used_fallback = core.safe_decide(BrokenAI(mode).vote, demo_view, demo_view.candidates, demo_state, "vote")
    print("%-8s -> 최종 선택: %s | fallback: %s | 재시도 +%d" % (
        mode, target, used_fallback, demo_state.stats["retries"] - before["retries"]))
print("\n통계:", demo_state.stats)

fuzzy    -> 최종 선택: 최민지 | fallback: False | 재시도 +0
garbage  -> 최종 선택: 오지환 | fallback: True | 재시도 +2
error    -> 최종 선택: 한소율 | fallback: True | 재시도 +2

통계: {'retries': 4, 'fallbacks': 2, 'leak_blocked': 0, 'llm_calls': 7}


In [13]:
print("===== 02 완료: 이제 03_mafia_game.ipynb 로 이동하세요 =====")
print("이 노트북에서 사용한 LLM 호출 수(프로세스 전체):", agents.LLM_CALLS["count"])

===== 02 완료: 이제 03_mafia_game.ipynb 로 이동하세요 =====
이 노트북에서 사용한 LLM 호출 수(프로세스 전체): 13
